# Attention Flow Analysis

Track how attention flows through the model: rollout across layers,
source token attribution, entropy evolution, and attention distance.

## Why This Matters

Attention flow analysis traces information paths through the network by composing attention patterns across layers. Flow paths, bottleneck detection, and source attribution reveal the multi-layer routing of information that single-layer attention patterns cannot capture.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_flow_analysis import (
    attention_rollout, source_token_attribution,
    layer_attention_entropy, attention_distance_profile,
    attention_flow_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Attention Rollout

Multiply attention patterns layer by layer to see end-to-end flow.

In [ ]:
result = attention_rollout(model, tokens)
rollout = result['rollout_matrix']
print(f"Rollout shape: {rollout.shape}")
print(f"Row sums (should be ~1): {jnp.sum(rollout, axis=-1)}")
print(f"\nRollout from last position to all sources:")
for i in range(rollout.shape[0]):
    bar = '█' * int(float(rollout[-1, i]) * 40)
    print(f"  pos {i}: {float(rollout[-1, i]):.4f} {bar}")

## Source Token Attribution

How much does each source token contribute to the final representation?

In [ ]:
result = source_token_attribution(model, tokens, target_position=-1)
print(f"Target position: {result['target_position']}")
for entry in result['per_source']:
    bar = '█' * int(entry['attribution'] * 40)
    print(f"  Source pos {entry['position']}: {entry['attribution']:.4f} {bar}")

## Layer Attention Entropy

How spread out is attention at each layer?

In [ ]:
result = layer_attention_entropy(model, tokens)
for p in result['per_layer']:
    bar = '█' * int(p['mean_entropy'] * 10)
    print(f"  Layer {p['layer']}: mean_H={p['mean_entropy']:.4f} "
          f"min={p['min_entropy']:.4f} max={p['max_entropy']:.4f} {bar}")

## Attention Distance Profile

How far does each head look on average?

In [ ]:
result = attention_distance_profile(model, tokens)
for p in result['per_layer']:
    for h in p['per_head']:
        dist_bar = '█' * int(h['mean_distance'] * 5)
        print(f"  L{p['layer']}H{h['head']}: dist={h['mean_distance']:.3f} "
              f"local={h['local_fraction']:.3f} {dist_bar}")

## Flow Summary

In [ ]:
result = attention_flow_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: entropy={p['mean_entropy']:.4f} "
          f"distance={p['mean_distance']:.4f}")